# Rung 1 — the content-based induction task (issue #200)

One Kaggle/Colab session, decided **before** seeing results. The
pre-registration is `07_capstone/notes/variable-distance-induction-preregistration.md`
(and its 2026-10-22, 2026-10-25 and 2026-11-01 addenda) and the design is
ADR-0033.

On the periodic task the previous occurrence of a token is always at `q-L`,
so "attend to `q-L+1`" and "attend to the token after the previous
occurrence" are the *same function* on the training distribution, and the
2026-10-19 probe showed the model had learned the offset. This task puts
the repeats at *varying* distances, so the fixed-offset shortcut does not
exist: the best single fixed offset explains a small fraction of the copy
queries (measured on the data before this run), while a content-matching
head can reach 100%.

| Change | Where | Why |
|---|---|---|
| **Vocabulary 2048 → 256** | `--content-standard` | The periodic task's 2048 answered a constraint this task does not have: its prefix is sampled *with replacement* (so a large vocabulary is what keeps the prefix collision-free), while the content task's key is sampled *without* replacement. The standard-geometry CPU probe of 2026-10-22 was still at the uniform prediction after 400 epochs at 2048. |
| **The verdict is a module, and it runs last** | Step 5 | The verdict used to live in a notebook cell that read `mean_path_patch_effect` out of the **exp1** manifest — a metric only the **exp4** manifest carries. It raised `KeyError` on the first seed, after the session was spent. The criteria now live in `src/experiments/content_verdict.py`, the exp4 manifest is read from where it is written, and the verdict cell is the **last** one: a verdict that crashes must not cost the session. |
| **The cascade is pinned to exp1's model** | Steps 2–3 | exp4 and exp5 now receive the epoch and the commit exp1 recorded, and exp5 receives the *model* seed separately from the SAE seed. A cascade about a stale checkpoint, another seed's weights or another commit's instrument is refused instead of silently measured. |
| **A distributed circuit is a secondary finding, not a verdict** | Step 5 | The 2026-10-25 CPU de-risk at the GPU geometry copied at 0.999 but confirmed no head: the copy was spread over a group. The seven criteria stay frozen; the circuit report describes the group. If the full run reproduces that shape, the verdict is still NEGATIVE and the report says which circuit carried the behaviour. |

The 2026-10-25 addendum removed the emergence run: its only unique output
was `confirmed_epoch`, which the full-budget run already records, and a
second early-stopped run would have doubled the GPU budget without adding a
measurement the verdict uses.

| Step | What runs | Why |
|------|-----------|-----|
| 0 | **Pilot**: 20 epochs of the content config, timed | Measures this GPU's real throughput (and the fresh-batch data generation, which is pure Python) so the session budget is planned from a measurement instead of an estimate |
| 1 | `exp1 --content-standard` | The **full-budget** measurement: 3000 epochs (384k optimizer steps per seed). The model it leaves behind is the one Rungs 4 and 5 analyse; the manifest records `confirmed_epoch` and writes one recovery manifest per finished seed |
| 2 | `exp4 --from-checkpoint … --source-epochs … --source-git-sha …` | Activation patching + head ablation + path patching **on the very model Rung 1 trained**, measured at a *copy query*, with the checkpoint's epoch and commit checked |
| 3 | `exp5 --activations-from … --source-model-seed …` | The SAE on real activations from the same model, on the best content-based seed when no head is confirmed |
| 4 | zip + download | From here on the artifacts cannot be lost |
| 5 | the verdict, by code | Seven frozen criteria, joined across the two manifests; the circuit report is secondary |

**The verdict is computed by code, not by hand.** A seed is never dropped and
the run is never repeated "to get a better label".


In [ ]:
BRANCH = "main"  # must contain the #200 instrument (content_verdict + checkpoint provenance)
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

# Fail before spending GPU time if the clone is stale or dirty. A dirty tree
# is refused by pin_remote_run.py, and a tree without these modules cannot
# compute the verdict this notebook exists to produce.
import subprocess

_status = subprocess.check_output(["git", "status", "--porcelain"], text=True)
assert not _status.strip(), f"the clone is dirty:\n{_status}"

from src.experiments.content_verdict import verdict_payload  # noqa: F401
from src.experiments.exp4_circuit_patching import build_parser as _exp4
from src.experiments.exp5_sae_dashboard import build_parser as _exp5

assert _exp4().parse_args(
    ["--from-checkpoint", "x", "--source-epochs", "1", "--source-git-sha", "abc"]
).source_epochs == 1, "exp4 is from before the provenance instrument"
assert _exp5().parse_args(["--source-model-seed", "0"]).source_model_seed == 0, (
    "exp5 is from before the source-model-seed instrument"
)
print("instrument present at", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# The GPU images already ship the CUDA torch stack plus numpy, matplotlib, tqdm and
# pyyaml. `uv sync --frozen` would re-resolve the ~5GB CUDA wheel set into the
# notebook's limited working disk and die mid-download — exactly how the first
# grokking kernel errored (2026-10-14). Use the preinstalled stack and fill only a
# real gap; the manifests record the torch/python versions actually used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}

import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])


## Step 0 — Pilot: measure this GPU before spending the session

Twenty epochs of the **content config** (same shape, same fresh-batch
regime, same detection cadence), timed. The output is the number the
budget has to be planned from: the old notebook estimated "~2h" per run,
which was never measured.

If the projection does not fit the session, the fix is **one seed per
session** (set `SEEDS` below to a single seed), never a shorter budget: a
truncated run is exactly how this repository produced under-trained false
negatives.


In [ ]:
# 20 epochs of the content config: 20 * 128 = 2560 optimizer steps.
# The flags are explicit (not --content-standard) because the pilot needs its
# own epoch count; everything else is the content config.
import os, time
t0 = time.time()
!python -m src.experiments.exp1_induction_heads --task variable --vocab-size 256 --seq-len 64 --d-model 64 --n-layers 2 --n-heads 4 --epochs 20 --batch-size 64 --num-train 8192 --batches fresh --seeds 0 --detect-every 10 --manifest-path results/probe_pilot_content_quick.json
pilot_seconds = time.time() - t0
steps = 20 * 128
rate = steps / max(pilot_seconds, 1e-6)
full = 3000 * 128
print(f"pilot: {pilot_seconds:.1f}s for {steps} steps -> {rate:.1f} steps/s")
print(f"projected standard run: {full / max(rate, 1e-6) / 60:.1f} min per seed")
print(f"projected 3 seeds:       {3 * full / max(rate, 1e-6) / 3600:.2f} h")


## Step 1 — The full-budget run (the measurement Rungs 4 and 5 use)

The content task's standard config (`--content-standard`: vocab 256, seq 64,
d_model 64, 2 layers, 4 heads, fresh batches, 3000 epochs = 384k optimizer
steps per seed), at its full budget, no early stop. The model this run
leaves behind is the one the circuit-patching and SAE rungs analyse.

The manifest records `confirmed_epoch`, so the emergence question — "when
does the head appear?" — is answered by the same run. The 2026-10-25
addendum removed the separate early-stopped run: it added no measurement
the verdict uses and would have doubled the GPU budget.


## Seeds

`SEEDS` is an environment variable so a short session can run one seed at
a time; the checkpoints and the per-seed manifest are the same either way.
Set it below. A one-seed session is a **pilot**: the run verdict needs the
three pre-registered seeds and reports `INCOMPLETE` until they are all
present.


In [ ]:
import os

os.environ["SEEDS"] = "0,1,2"  # set to a single seed (e.g. "0") if the session is short


In [ ]:
!python -m src.experiments.exp1_induction_heads --content-standard --seeds $SEEDS --checkpoint-dir checkpoints/induction_content --checkpoint-every 250 --resume --detect-every 50 --manifest-path results/induction_content.json


## Step 2 — Rung 4 on the very model Rung 1 trained

The checkpoint is loaded, not retrained. That is what makes the cascade one
model instead of two, and `--source-task variable` makes it analyse
variable-distance data and measure the interventions at a copy query.

**The cascade runs when the *behaviour* is content-based**, not only when a
per-head detector fires: a copy spread over several heads is invisible to
the 0.30 screen in every one of them, and the group intervention exists
precisely for that case.

`--counterfactual minimal-pair` builds the corrupted half of the
intervention by changing **one token** of the same sequence — the token at
the source of the copy — instead of taking a different sequence. The
2026-10-20 cascade compared a sequence against an unrelated one, which is
why its activation-patching recovery (0.0834) is hard to read as "this
circuit carries the answer".


In [ ]:
# If the session died before the last seed, the multi-seed manifest is absent.
# Rebuild it from the per-seed recovery manifests before Rungs 4 and 5 read it.
# The verdict will still be INCOMPLETE unless seeds 0, 1 and 2 are all present;
# this cell only makes the partial session's cascade possible, it does not
# turn a partial run into a verdict.
import subprocess
import sys
from pathlib import Path

full = Path("results/induction_content.json")
if not full.exists():
    per_seed = sorted(Path("results").glob("induction_content_seed*.json"))
    if not per_seed:
        raise SystemExit(
            "no multi-seed manifest and no per-seed recovery manifests: "
            "the exp1 cell did not finish a seed"
        )
    seeds = sorted(int(p.stem.rsplit("seed", 1)[1]) for p in per_seed)
    print("rebuilding", full, "from", [p.name for p in per_seed])
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.experiments.merge_manifests",
            "--inputs",
            *[str(p) for p in per_seed],
            "--output",
            str(full),
            "--expected-seeds",
            ",".join(str(seed) for seed in seeds),
            "--allow-optional",
        ],
        check=True,
    )
    print("recovered seeds:", seeds)
else:
    print("multi-seed manifest present:", full)


In [ ]:
import json

m_full = json.load(open("results/induction_content.json"))

BEHAVIOUR_IS_CONTENT = any(
    float(m["copy_accuracy"]) - float(m["fixed_offset_baseline"]) >= 0.20
    for m in m_full["per_seed_metrics"]
)
FOUND_HEAD = any(
    int(m["total_induction_heads"]) > 0 for m in m_full["per_seed_metrics"]
)
print("content-based behaviour:", BEHAVIOUR_IS_CONTENT, "| confirmed head:", FOUND_HEAD)

# The cascade must analyse the final model exp1 reported, not a rolling
# checkpoint that lagged it, and not weights from another commit. Both facts
# come from exp1's own manifest, so the flags cannot drift from the run.
EXP1_EPOCHS = {int(m["epochs_run"]) for m in m_full["per_seed_metrics"]}
assert len(EXP1_EPOCHS) == 1, f"seeds ran different budgets: {sorted(EXP1_EPOCHS)}"
EXP1_EPOCHS = EXP1_EPOCHS.pop()
EXP1_SHA = m_full["git_sha"]

if BEHAVIOUR_IS_CONTENT or FOUND_HEAD:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_content --source-vocab-size 256 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --source-epochs {EXP1_EPOCHS} --source-git-sha {EXP1_SHA} --counterfactual minimal-pair --seeds $SEEDS --manifest-path results/exp4_content.json
else:
    print("Neither a confirmed head nor a content-based behaviour: Rung 4 has no")
    print("target and is skipped. The verdict in Step 5 says so explicitly.")


In [ ]:
# The cascade's numbers live in the exp4 manifest — read them from there.
# (The old notebook read them out of the exp1 manifest, which never carried
# them: that is the KeyError the verdict module replaced.)
# Path patching and ablation are omitted when no head was confirmed — the
# instrument must not report a fabricated 0.0 — so read them with .get().
import json
from pathlib import Path


def _fmt(metrics, key):
    value = metrics.get(key)
    return "not measured" if value is None else f"{float(value):.3f}"


if Path("results/exp4_content.json").exists():
    m_exp4 = json.load(open("results/exp4_content.json"))
    for seed, metrics in zip(m_exp4["seeds"], m_exp4["per_seed_metrics"]):
        print(f"exp4 seed {seed}: heads={int(metrics['total_induction_heads'])} "
              f"(candidates {int(metrics['n_candidate_heads'])}) | "
              f"ablation {_fmt(metrics, 'mean_ablation_effect')} | "
              f"path patch {_fmt(metrics, 'mean_path_patch_effect')} | "
              f"group path patch {_fmt(metrics, 'group_path_patch_effect')} | "
              f"activation recovery {metrics['mean_activation_recovery']:.3f} | "
              f"checkpoint epoch {int(metrics['checkpoint_epoch'])}")
else:
    print("no exp4 manifest: the cascade did not run")


## Step 3 — Rung 5: the SAE on real activations

The same model, the same data, the same checkpoint: the SAE is trained on
the activations Rung 1's model actually produces, and it is skipped (not
faked) when no seed confirmed a head.


In [ ]:
import glob
import json
import os


def seed_for_cascade(manifest):
    """The seed whose checkpoint Rungs 4 and 5 should analyse.

    A confirmed head is the primary target. When the screen confirms none,
    a *content-based behaviour* is still the pre-specified secondary
    finding: the copy exists, and it is spread over a group of heads. In
    that case the SAE harvests the strongest content-based seed instead of
    skipping the rung — the activations of a model that copies are exactly
    what the dictionary is meant to describe.
    """
    m = json.load(open(manifest))
    rows = list(zip(m["seeds"], m["per_seed_metrics"]))
    confirmed = [seed for seed, metrics in rows if int(metrics["total_induction_heads"]) > 0]
    if confirmed:
        return confirmed[0], "confirmed head"
    scored = sorted(
        (
            float(metrics["copy_accuracy"]) - float(metrics["fixed_offset_baseline"]),
            seed,
        )
        for seed, metrics in rows
    )
    excess, seed = scored[-1]
    if excess >= 0.20:
        return seed, f"distributed content circuit (excess {excess:.3f})"
    return None, "neither a confirmed head nor content-based behaviour"


seed, why = seed_for_cascade("results/induction_content.json")
ckpt_dir = "checkpoints/induction_content"

if seed is None:
    print(f"No seed can anchor Rung 5: {why}. It is skipped, not faked.")
else:
    ckpt = glob.glob(f"{ckpt_dir}/*seed{seed}*.pt")
    assert ckpt, f"no checkpoint for model seed {seed} in {ckpt_dir}"
    print(f"harvesting model seed {seed} ({why}) from {ckpt}")
    !python -m src.experiments.exp5_sae_dashboard --activations-from {ckpt[0]} --source-vocab-size 256 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --source-model-seed {seed} --source-epochs {EXP1_EPOCHS} --source-git-sha {EXP1_SHA} --seeds $SEEDS --manifest-path results/exp5_content.json


## Step 4 — Zip and download

The manifests are zipped **before** the verdict, so a verdict that fails
cannot cost the session's artifacts.


In [ ]:
# Collect every manifest this session produced, then download.
# This cell runs BEFORE the verdict on purpose: the verdict is the last
# thing that can fail, and it must not be able to cost the session.
# The per-seed recovery manifests are included: if the session died between
# seeds, scripts/merge_manifests.py rebuilds the multi-seed record from them.
import glob

from scripts.zip_manifests import write_manifest_zip

manifests = sorted(
    set(
        glob.glob("results/induction_content*.json")
        + glob.glob("results/exp4_content.json")
        + glob.glob("results/exp5_content.json")
    )
)
manifests = [p for p in manifests if "probe_" not in p]
assert manifests, "no manifests produced - nothing to zip"
print("manifests:", manifests)

# The old shell-zip line expanded a Python list to its repr in IPython,
# which is not a valid shell argument. The helper writes the archive
# directly, so there is no shell quoting to get wrong (2026-11-04 fix).
archive = write_manifest_zip(manifests, "induction_content_results.zip")
print("wrote", archive, "with", len(manifests), "manifest(s)")

try:
    from google.colab import files

    files.download("induction_content_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")


## Step 5 — The verdict, computed from the manifests

The seven pre-registered criteria, in the order they were frozen. A
criterion is met when the *seed* meets it; the run verdict requires the
three pre-registered seeds (0, 1, 2). With fewer seeds the session is a
**pilot** and reports `INCOMPLETE` — never a POSITIVE — and a manifest
whose criteria are missing reports `INCOMPLETE` too. When all three seeds
are present, the run is POSITIVE when a majority meet all seven, PARTIAL
when a head is confirmed but the causal criteria fail, and NEGATIVE
otherwise.

The criteria live in `src/experiments/content_verdict.py` — the same module
the test suite checks — so the verdict cannot drift from the measurement
and cannot crash on a metric that lives in the other manifest. The
mechanism report printed next to the verdict (the aggregate content mass,
the group ablation, the attribution) is **secondary**: it describes the
circuit, it does not decide the verdict.


In [ ]:
from pathlib import Path

from src.experiments.content_verdict import (
    format_report,
    load_manifest,
    per_seed_metrics,
    run_verdict,
    verdict_payload,
    write_verdict_json,
)


def report(exp1_path, exp4_path=None):
    """The pre-registered verdict of one run, joined across the manifests."""
    exp1 = load_manifest(exp1_path)
    exp4 = None
    if exp4_path and Path(exp4_path).exists():
        exp4 = load_manifest(exp4_path)
    result = run_verdict(exp1, exp4)
    exp1_rows = per_seed_metrics(exp1)
    exp4_rows = per_seed_metrics(exp4) if exp4 is not None else {}
    payload = verdict_payload(
        result,
        exp1_rows,
        exp4_rows,
        exp1.get("args", {}).get("task"),
        {"exp1": exp1_path, "exp4": exp4_path},
    )
    out = write_verdict_json(payload, "results/content_verdict.json")
    print(f"\n=== {exp1_path} ===")
    print(format_report(result, exp1_rows, exp4_rows))
    print(f"verdict written to {out}")
    return result


full = report("results/induction_content.json", "results/exp4_content.json")


In [ ]:
# The verdict JSON is an artifact of the session but not a ResultsManifest:
# it is never pinned as evidence. Put it next to the zip so the decision
# can be read without re-running the module.
import os

from scripts.zip_manifests import append_to_zip

if os.path.exists("results/content_verdict.json"):
    append_to_zip(["results/content_verdict.json"], "induction_content_results.zip")
    try:
        from google.colab import files

        files.download("results/content_verdict.json")
    except ImportError:
        print("Kaggle: results/content_verdict.json is in /kaggle/working.")
else:
    print("no verdict artifact (the verdict cell did not reach the write)")


## After downloading

1. If the session died before the last seed, rebuild the multi-seed record
   from the per-seed recovery manifests and check the pre-registered seed
   set:
   `python -m src.experiments.merge_manifests --inputs <dir> --output results/induction_content.json --expected-seeds 0,1,2 --allow-optional`
   (`--allow-optional` is for a metric that is legitimately undefined for a
   seed, e.g. no confirmed head; the per-seed rows stay the source of truth).
2. Unzip into a scratch dir, then pin **only the records**, naming exactly
   the files the session produced:
   `python scripts/pin_remote_run.py --from <dir> --only induction_content.json,exp4_content.json,exp5_content.json`
   (`exp5_content.json` is required only when Step 3 ran; the pinner fails
   closed on any named file that is missing, so omit it if Step 3 was
   skipped.) Without `--only` the tool fails closed on per-seed manifests
   and on `content_verdict.json`, which are valid JSON but not the record.
3. `make verify-claims` must stay green — the new manifests need their
   `<!-- manifest: results/... -->` tags next to the numbers they back.
4. Report exactly what happened, including "no content-based head at the
   full budget": the answer to issue #200 is data, either way. The
   pre-registration says a negative is a negative. The saved
   `content_verdict.json` records the decision and the per-seed checks; it
   is not pinned as evidence because it is not a `ResultsManifest`.
5. Update the Rung 1 section of `portfolio/RESULTS.md` with the new
   numbers, per seed, and with the verdict the criteria produced.
6. If the verdict is NEGATIVE but the mechanism report describes a
   distributed circuit, **do not change the headline**. The seven frozen
   criteria remain the verdict. Report the circuit as the pre-specified
   **secondary finding** in the mechanism section, with the group ablation
   and the exp4 group path-patch effect that measured it — and say that no
   single head passed the screen, which is why the primary verdict is
   NEGATIVE.
